In [ ]:
from pathlib import Path




# Define the types of networks as an enumeration for type safety and clarity.
class NetworkType(Enum):
    LF = "LF"
    MF = "MF"
    HF = "HF"
    HFLIN = "Hflin"
    HFPER = "Hfper"
    INTER = "Inter"
    LSTM = "LSTM"
    STEP = "step"

# Factory class to build different types of networks based on the provided type.
class NetworkFactory:

    @staticmethod
    def build_network(network_type: str,
                      names: List[str] = [],
                      params: Optional[dict] = None,
                      data_train: Optional[Union[np.ndarray, List[np.ndarray]]] = None,
                      output_train: Optional[Union[np.ndarray, List[np.ndarray]]] = None,
                      N: int = 1000,
                      n: int = 10,
                      train: bool = True,
                      do_HPO: bool = False,
                      verbose: bool = False) -> 'INetwork':
        """
        Build and return a network of the specified type.

        Parameters:
        - network_type (str): Type of the network to be built.
        - names (List[str]): List of names used for MultiFidelity networks.
        - params (Optional[dict]): Dictionary of parameters for the network.
        - data_train (Optional[Union[np.ndarray, List[np.ndarray]]]): Training data.
        - output_train (Optional[Union[np.ndarray, List[np.ndarray]]]): Training output data.
        - N (int): Number of samples.
        - n (int): Some integer parameter.
        - train (bool): Flag indicating whether to train the network.
        - do_HPO (bool): Flag indicating whether to perform hyperparameter optimization.
        - verbose (bool): Flag indicating whether to print verbose output.

        Returns:
        - INetwork: The created network object.

        Raises:
        - ValueError: If an invalid network type is provided.
        """
        try:
            # Use Enum for safer and clearer type checking.
            network_type_enum = NetworkType[network_type.upper()]
        except KeyError:
            raise ValueError(f"Invalid network type: {network_type}")

        # Map the enum to the respective network class constructors.
        if network_type_enum in {NetworkType.LF, NetworkType.MF, NetworkType.HF, NetworkType.HFLIN, NetworkType.HFPER}:
            return Neural_Network(network_type, params, data_train, output_train, N, n, train, do_HPO, verbose)
        
        elif network_type_enum == NetworkType.STEP:
            # Ensure data_train and output_train are lists for MultiFidelity networks.
            if not (isinstance(data_train, list) and isinstance(output_train, list)):
                raise ValueError("For MultiFidelity network, data_train and output_train must be lists of numpy arrays.")
            return MultiFidelity(names, params, data_train, output_train, N, n, do_HPO, verbose)
        
        elif network_type_enum == NetworkType.INTER:
            return Intermediate(params, data_train, output_train, N, n, train, do_HPO, verbose)
        
        elif network_type_enum == NetworkType.LSTM:
            return LSTM(params, data_train, output_train, N, train, do_HPO, verbose)

        raise ValueError(f"Invalid network type: {network_type}")

# Abstract base class for network-related operations.
class INetwork(ABC):
    """
    Abstract base class for defining a network interface with essential methods.
    This class uses the Abstract Base Class (ABC) mechanism to enforce the implementation
    of specific methods in any subclass.
    """

    def __init__(self):
        self.inputs = None
        self.transformations = []

    def variable_input(self, input_discr: Any) -> None:
        """
        Sets the input variable when solving the inverse problem.
        
        Args:
            input_discr (Any): The input discriminator.
        """
        self.inputs = input_discr

    def _input_wrapper_prediction(self, x_test: np.ndarray, multi_input: bool = False) -> np.ndarray:
        """
        Prepare input data for prediction.

        Parameters:
        - x_test (np.ndarray): Test data for prediction.
        - multi_input (bool): Flag indicating if there are multiple inputs.

        Returns:
        - np.ndarray: Prepared data for prediction or an empty array if inputs are not set.
        """
        # Ensure x_test is 2D.
        if x_test.ndim == 1:
            x_test = x_test.reshape(-1, 1)

        # Transpose x_test for consistent shape.
        x_test = x_test.T

        # Apply transformations if any.
        if self.transformations:
            x_final = reduce(lambda acc, transf: np.hstack([acc, transf(acc)]), self.transformations, x_test)
        else:
            x_final = x_test

        # Handle inputs and prepare final data for prediction.
        if self.inputs is not None:
            x_final = np.tile(x_final, (self.inputs.shape[0], 1))
        else:
            warning_message = "Inputs are not set."
            warnings.warn(warning_message, UserWarning)
            return np.array([])

        return x_final

    def wrapper_prediction(self, x_test: np.ndarray, multi_input: bool = False) -> np.ndarray:
        """
        Wrapper for making predictions with processed test inputs.
        
        Args:
            x_test (np.ndarray): Test input data.
            multi_input (bool): Flag to indicate if multiple inputs are used.
        
        Returns:
            np.ndarray: Predicted output data.
        """
        x_final = self._input_wrapper_prediction(x_test, multi_input)
        return self.prediction(np.concatenate((self.inputs, x_final), axis=1)).flatten()      

    @compute_time
    def param_inverse(self, mean_prior: np.ndarray, x_data: np.ndarray, cov_prior: Optional[np.ndarray] = None, 
                      cov_noise: float = 0.1, cov_likelihood: Optional[np.ndarray] = None, y_obs: Optional[np.ndarray] = None, 
                      x_real: Optional[np.ndarray] = None, number_chains: int = 1, N: int = 1000, burn_in: int = 500, 
                      levels: int = 1, diagnostic: bool = True, rwmh_cov: Optional[np.ndarray] = None, rmwh_scaling: float = 0.1, 
                      rwmh_adaptive: bool = True, algo: str = "MH", transformation: List[Any] = []) -> Tuple[np.ndarray, np.ndarray]:

        """
        Perform parameter inversion using MCMC sampling.

        Parameters:
        - mean_prior: Mean of the prior distribution.
        - x_data: Input data.
        - cov_prior: Covariance of the prior distribution (optional).
        - cov_noise: Noise covariance.
        - cov_likelihood: Covariance of the likelihood (optional).
        - y_obs: Observed data (optional).
        - x_real: Real parameters (optional).
        - number_chains: Number of MCMC chains.
        - N: Number of MCMC iterations.
        - burn_in: Number of burn-in iterations.
        - levels: Number of model levels.
        - diagnostic: Flag to enable diagnostic plots.
        - rwmh_cov: Covariance matrix for RWMH proposal (optional).
        - rmwh_scaling: Scaling factor for RWMH.
        - rwmh_adaptive: Flag for adaptive RWMH.
        - algo: MCMC algorithm to use ("MH", "AM", "CN", "DREAMZ").
        - transformation: List of transformations to apply.

        Returns:
        - estimates: MCMC estimates of the parameters.
        - error: Relative error of the estimates.
        """
        self.transformations = transformation
        self.inputs = x_data

        if x_real is not None:
            dim = x_real.shape[0]
        elif y_obs is not None:
            dim = y_obs.shape[0]
        else:
            warnings.warn("No observation nor data given", UserWarning)
            return np.array([]), np.array([])

        if N <= burn_in:
            warnings.warn("Number of steps insufficient, smaller or equal to burn-in", UserWarning)

        if cov_prior is None:
            cov_prior = mean_prior * 0.2

        if cov_likelihood is None:
            cov_likelihood = cov_noise**2 * np.eye(x_real.shape[0])

        my_prior = multivariate_normal(mean_prior, cov_prior)

        if y_obs is None:
            y_obs = self.wrapper_prediction(x_real) + np.random.normal(loc=0., scale=cov_noise, size=x_real.shape)
        else:
            y_obs += np.random.normal(loc=0., scale=cov_noise, size=y_obs.shape)
            y_obs = y_obs.flatten()

        if levels > 1:
            if levels > len(self.model_list):
                warnings.warn("Number of levels is exceeding the number of models", UserWarning)
            else:
                my_loglike = [tda.GaussianLogLike(y_obs, cov_likelihood) for _ in range(levels)]
                my_posterior = [tda.Posterior(my_prior, my_loglike[i], self.model_list[i].wrapper_prediction) for i in range(levels)]
        else:
            my_loglike = tda.GaussianLogLike(y_obs, cov_likelihood)
            my_posterior = tda.Posterior(my_prior, my_loglike, self.wrapper_prediction)

        if rwmh_cov is None:
            rwmh_cov = np.eye(len(x_real))

        estimates = MCMC(my_posterior, N, burn_in, number_chains, diagnostic, rwmh_cov, rmwh_scaling, rwmh_adaptive, algo, dim)

        if diagnostic:
            plot_hist(estimates, x_real, self.wrapper_prediction(estimates), self.wrapper_prediction(x_real))
        
        error = np.abs(estimates - x_real) / np.abs(x_real + 1e-10)
        return estimates, error    

    @compute_time
    def inverse_cuqi(mean_prior: np.ndarray, 
                    x_data: np.ndarray,
                    x_real: Optional[np.ndarray] = None, 
                    y_obs: Optional[np.ndarray] = None, 
                    N: int = 1000, 
                    burn_in: int = 500, 
                    cov_prior: float = 0.5, 
                    sd_noise: float = 0.1,
                    adapt: bool = False, 
                    scale: float = 0.3, 
                    proposal_sd: float = 0.3, 
                    x_init: Optional[Union[int, float, np.ndarray]] = None, 
                    diagnostic: bool = True, 
                    number_chains: int = 1, 
                    algo: str = "MH", 
                    transformation: List = []) -> Union[np.ndarray, float]:
        """
        Perform Bayesian inference using the CUQI framework.

        Parameters:
        - mean_prior: np.ndarray : Prior mean
        - x_data: np.ndarray : Input data
        - x_real: Optional[np.ndarray] : Real data (optional)
        - y_obs: Optional[np.ndarray] : Observed data (optional)
        - N: int : Number of iterations (default: 1000)
        - burn_in: int : Number of burn-in steps (default: 500)
        - cov_prior: float : Covariance of the prior (default: 0.5)
        - sd_noise: float : Standard deviation of noise (default: 0.1)
        - adapt: bool : Whether to adapt the proposal distribution (default: False)
        - scale: float : Scaling factor for the proposal distribution (default: 0.3)
        - proposal_sd: float : Standard deviation of the proposal distribution (default: 0.3)
        - x_init: Optional[Union[int, float, np.ndarray]] : Initial guess (default: None)
        - diagnostic: bool : Whether to plot diagnostic information (default: True)
        - number_chains: int : Number of MCMC chains (default: 1)
        - algo: str : Algorithm to use ("MH" or "NUTS", default: "MH")
        - transformation: List : List of transformations (default: [])

        Returns:
        - estimates: np.ndarray : Estimated parameters
        - error: float : Error with respect to true parameters
        """

        # Set inputs and transformations
        self.inputs = x_data
        self.transformations = transformation

        # Check if observations are provided
        if y_obs is not None:
            dim = y_obs.shape[0]
        else:
            warning_message = "No observation nor data given"
            warnings.warn(warning_message, UserWarning)
            return

        # Check if the number of steps is greater than burn-in period
        if N <= burn_in:
            warning_message = "Number of steps insufficient, smaller or equal than burn-in"
            warnings.warn(warning_message, UserWarning)
            return

        # Initialize x_init if not provided
        if x_init is None:
            x_init = np.random.rand(dim)
        elif isinstance(x_init, (int, float)):
            x_init = x_init * np.ones(dim)
        
        m = x_real.shape[0]
        
        # Select algorithm and initialize CuqiModel and Gaussian objects
        if algo == "NUTS":
            fun = Function(wrapper_prediction)
            A = CuqiModel(forward=wrapper_prediction, jacobian=fun.compute_jacobian, range_geometry=Continuous1D(dim), domain_geometry=Continuous1D(dim))
            x = Gaussian(mean=mean_prior, cov=cov_prior)
        else:
            A = CuqiModel(forward=wrapper_prediction, range_geometry=Continuous1D(dim), domain_geometry=Continuous1D(m))
            x = Gaussian(mean=mean_prior, cov=cov_prior)

        y = Gaussian(mean=A(x), cov=proposal_sd)

        # Generate or perturb observations
        if y_obs is None:
            y_obs = y(x=x_real).sample()
        else:
            y_obs = y_obs + np.random.normal(loc=0., scale=sd_noise, size=y_obs.shape)

        # Run MCMC to get estimates
        estimates = MCMC_cuqi(y, x, y_obs, N, burn_in, number_chains, diagnostic=diagnostic, algo=algo, adapt=adapt, scale=scale)
        estimates = np.mean(estimates, axis=1)

        # Calculate and print error
        error = np.linalg.norm(estimates - x_real)
        print(f"Error wrt true parameters: {error}")

        # Plot diagnostics if required
        if diagnostic:
            plot_hist(estimates, x_real, wrapper_prediction(estimates), wrapper_prediction(x_real))

        return estimates, error


    @abstractmethod
    def prediction(self) -> None:
        """
        Abstract method to be implemented for making predictions using the network.
        Subclasses must provide the implementation for this method.
        """
        pass

    @abstractmethod
    def performance(self) -> None:
        """
        Abstract method to be implemented for evaluating the network's performance.
        Subclasses must provide the implementation for this method.
        """
        pass

    @abstractmethod
    def HPO(self) -> None:
        """
        Abstract method to be implemented for hyperparameter optimization.
        Subclasses must provide the implementation for this method.
        """
        pass

    @abstractmethod
    def training(self) -> None:
        """
        Abstract method to be implemented for training the network.
        Subclasses must provide the implementation for this method.
        """
        pass

    @staticmethod
    @abstractmethod
    def save() -> None:
        """
        Abstract static method to be implemented for saving the network's state or model.
        Subclasses must provide the implementation for this method.
        """
        pass



In [ ]:
class Neural_Network(INetwork):
    
    def __init__(
        self, 
        name: str, 
        params: Optional[dict] = None, 
        data_train: Optional[np.ndarray] = None, 
        output_train: Optional[np.ndarray] = None, 
        N: int = 1000, 
        n: int = 10, 
        train: bool = True, 
        do_HPO: bool = False, 
        transformations: Optional[list] = None, 
        verbose: bool = False
    ):
        """
        Initializes the Neural_Network instance.
        
        Args:
            name (str): Name of the network.
            params (Optional[dict]): Hyperparameters of the network.
            data_train (Optional[np.ndarray]): Training data.
            output_train (Optional[np.ndarray]): Training outputs.
            N (int): Number of epochs for training.
            n (int): Batch size for training.
            train (bool): Flag to indicate if training should be performed.
            do_HPO (bool): Flag to indicate if hyperparameter optimization is to be performed.
            transformations (Optional[list]): List of transformations to apply to the data.
            verbose (bool): Flag to indicate verbosity of the output.
        """
        K.clear_session()
        self.name = name
        self.params = params
        self.N = N
        self.n = n
        self.verbose = verbose
        self.hist = None
        self.data_train = data_train
        self.output_train = output_train
        self.transformations = transformations if transformations is not None else []
        self.inputs = None

        # Set input and output shapes based on training data dimensions
        self.input_shape = self._get_shape(data_train)
        self.output_shape = self._get_shape(output_train)
        # Perform hyperparameter optimization if required or if no parameters are provided
        if do_HPO or params is None:
            if output_train is None or data_train is None:
                warning_message = "Not enough data given!"
                warnings.warn(warning_message, UserWarning)
            self.params = self.HPO(data_train, output_train)

        # Initialize the model
        self.model = getModel(self.params, self.input_shape, self.name, self.output_shape)

        # Train the model if required
        if train:
            self.hist = self.training(data_train, output_train, epoch=self.N, batch=self.n) 
            self.plot_training_loss()

    def _get_shape(self, data: Optional[np.ndarray]) -> int:
        """
        Gets the shape of the data.

        Args:
            data (Optional[np.ndarray]): The data to get the shape of.

        Returns:
            int: The shape of the data.
        """
        return data.shape[1] if data is not None and len(data.shape) > 1 else 1


    def plot_training_loss(self) -> None:
        """
        Plots the training loss.
        """
        if self.hist is not None and 'loss' in self.hist.history:
            plt.plot(self.hist.history['loss'][100:], label='Training Loss')
            plt.title('Mean Squared Error (MSE) over Epochs')
            plt.xlabel('Epochs')
            plt.ylabel('MSE')
            plt.legend()
            plt.show()
        else:
            warnings.warn("No training history or 'loss' key found.", UserWarning)


    @compute_time
    def training(self, x: np.ndarray, y: np.ndarray, epoch: int, batch: int) -> Any:
        """
        Trains the model on the given data.
        
        Args:
            x (np.ndarray): Training data.
            y (np.ndarray): Training outputs.
            epoch (int): Number of epochs for training.
            batch (int): Batch size for training.

        Returns:
            Any: The training history.
        """
        self.hist = self.model.fit(x, y, epochs=epoch, batch_size=batch, verbose=0)
        return self.hist


    def prediction(self, x_test: np.ndarray) -> np.ndarray:
        """
        Makes predictions on the test data.

        Args:
            x_test (np.ndarray): Test data.

        Returns:
            np.ndarray: Predicted values.
        """
        if self.verbose:
            return self.model.predict(x_test)
        else:
            with Suppressor():
                return self.model.predict(x_test)


    def HPO(self, data_train: np.ndarray, output_train: np.ndarray) -> Dict[str, Any]:
        """
        Performs hyperparameter optimization using Bayesian optimization.

        Args:
            data_train (np.ndarray): Training data.
            output_train (np.ndarray): Training outputs.

        Returns:
            Dict[str, Any]: The best hyperparameters found.
        """
        def objective(trial):
            K.clear_session()
            params = {
                "nodes": trial.suggest_int("nodes", 4, 64, log=True),
                "l2weight": trial.suggest_float("l2weight", 1e-4, 1e-1, log=True),  # Adjusted to use suggest_float
                "lr": trial.suggest_float("lr", 1e-4, 1e-1, log=True),  # Adjusted to use suggest_float
                "kernel_init": trial.suggest_categorical("kernel_init", ["uniform", "glorot_uniform"]),
                "opt": trial.suggest_categorical("opt", ["Adam", "Adamax"]),

            }
            loss = kCrossVal(self.n, self.N, data_train, output_train, params, self.name, self.input_shape, self.output_shape)
            return loss

        study = optuna.create_study(direction="minimize")
        study.optimize(objective, n_trials=2, n_jobs=-1)  # Parallelize trials
        best_params = study.best_params
        return best_params

    def objective(self, params: Dict[str, Any]) -> Dict[str, Any]:
        """
        Objective function to minimize during hyperparameter optimization.

        Args:
            params (Dict[str, Any]): Hyperparameters to evaluate.

        Returns:
            Dict[str, Any]: The loss value and parameters.
        """
        K.clear_session()
        loss = kCrossVal(self.n, self.N, self.data_train, self.output_train, params, self.name, self.input_shape, self.output_shape)
        return {"loss": loss, "params": params, "status": STATUS_OK}


    def performance(self, data_test: np.ndarray, output_test: np.ndarray) -> Tuple[float, float]:
        """
        Evaluate the performance of the model on test data.

        Args:
            data_test (np.ndarray): Test data.
            output_test (np.ndarray): Expected output data.

        Returns:
            Tuple[float, float]: Test Mean Squared Error (MSE) and R^2 score.
        """
        pred = self.prediction(data_test)

        if len(output_test.shape) < len(pred.shape):
            output_test = output_test[:, _]
        
        test_mse = np.mean(np.square(output_test - pred))
        print(f"Test MSE: {test_mse}")

        r2 = 1 - np.sum(np.square(output_test - pred)) / np.sum(
            np.square(output_test - np.mean(output_test))
        )
        print(f"R^2: {r2}")
        
        return test_mse, r2   


    def save(self, path: str, identifier: str="_") -> None:
        """
        Save the NN model and the class instance.

        Args:
            path (str): Directory path to save the model and instance.
            identifier (str): Identifier for the saved files.
        """
        # Ensure the directory exists
        os.makedirs(path, exist_ok=True)

        # Save the Keras model separately
        model_path = os.path.join(path, f'NN_model_{identifier}.h5')
        self.model.save(model_path)

        # Save the class instance excluding the Keras model
        temp_model = self.model
        self.model = None
        with open(os.path.join(path, f'class_instance_{identifier}.pkl'), 'wb') as f:
            pickle.dump(self, f)

        # Restore the model attribute
        self.model = temp_model

    @classmethod
    def load(cls, path: str, identifier: str) -> ' Neural_Network':
        """
        Load the LSTM model and the class instance.

        Args:
            path (str): Directory path from which to load the model and instance.
            identifier (str): Identifier for the saved files.

        Returns:
             Neural_Network: The loaded  Neural_Network instance.
        """
        with open(os.path.join(path, f'class_instance_{identifier}.pkl'), 'rb') as f:
            instance = pickle.load(f)

        model_path = os.path.join(path, f'NN_model_{identifier}.h5')
        custom_objects = {
            'mse': MeanSquaredError()  # Add any custom objects required by the model
        }
        instance.model = load_model(model_path, custom_objects=custom_objects)

        return instance



In [ ]:
class MultiFidelity(INetwork):
        
    def __init__(self, 
                 names: List[str], 
                 params: Optional[List[dict]] = None, 
                 data_train: Optional[List[np.ndarray]] = None, 
                 output_train: Optional[List[np.ndarray]] = None, 
                 N: Optional[List[int]] = None, 
                 n: Optional[List[int]] = None, 
                 do_HPO: bool = False, 
                 verbose: bool = False):
        """
        Initialize MultiFidelity network.

        Args:
            names (List[str]): Names of the networks.
            params (Optional[List[dict]]): Parameters for each network.
            data_train (Optional[List[np.ndarray]]): Training data for each network.
            output_train (Optional[List[np.ndarray]]): Training outputs for each network.
            N (Optional[List[int]]): Number of epochs for each network.
            n (Optional[List[int]]): Batch sizes for each network.
            do_HPO (bool): Whether to perform hyperparameter optimization.
            verbose (bool): Whether to print verbose output.
        """
        self.names = names
        self.Ns = N if N is not None else [1000] * len(names) 
        self.ns = n if n is not None else [10] * len(names)  
        self.data_train = data_train
        self.output_train=output_train
        self.steps = int((len(names) - 1) / (len(data_train) - 1)) + 1
        self.model_list = []
        self.transformations = []
        self.input_shape = 1
        self.output_shape = 1

        if not data_train or not output_train or len(data_train) != len(output_train):
            raise ValueError('The data are incoherent or insufficient')

        if data_train and len(data_train[0].shape) > 1:
            self.input_shape = data_train[0].shape[1]

        if output_train and len(output_train[0].shape) > 1:
            self.output_shape = output_train[0].shape[1]

        K.clear_session()
        if params is None:
            params = [None] * len(names)
        elif len(params) < len(names):
            params += [None] * (len(names) - len(params))

        count = 1
        for index, name in enumerate(names):
            model = NetworkFactory.build_network(
                name,
                params=params[index],
                data_train=data_train[count - 1],
                output_train=output_train[count - 1],
                N=self.Ns[index],
                n=self.ns[count - 1],
                train=True,
                do_HPO=do_HPO,
                verbose=verbose
            )
            self.model_list.append(model)
            
            if (index + 1) == (self.steps - 1) * (count - 1) + 1:
                count += 1

            for i in range(count - 1, len(data_train)):
                data_train[i] = np.c_[data_train[i], model.prediction(data_train[i])]


    def plot_training_loss(self) -> None:
        """
        Plot training loss for all models in the MultiFidelity network.
        """
        for model in self.model_list:
            model.plot_training_loss()


    
    # FUNZIONE PER TRAINING
    # FUNZIONE PER HPO

    def performance(self, data_test: np.ndarray, output_test: np.ndarray, position: Optional[int] = None) -> Tuple[float, float]:
        """
        Evaluate the performance of the model.

        Args:
            data_test (np.ndarray): Test data.
            output_test (np.ndarray): Expected output data.
            position (Optional[int]): Position of the model in the model list to evaluate.

        Returns:
            Tuple[float, float]: Test Mean Squared Error (MSE) and R^2 score.
        """
        if position is None:
            position = len(self.model_list)
        elif not isinstance(position, int) or position > len(self.model_list):
            raise ValueError('The required NN is not existent')

        data = copy.copy(data_test)

        for i in range(position - 1):
            data = np.c_[data, self.model_list[i].prediction(data).reshape(-1, 1)]

        pred = self.model_list[position - 1].prediction(data)

        if len(output_test.shape) < len(pred.shape):
            output_test = output_test[:, np.newaxis]

        test_mse = np.mean(np.square(output_test - pred))
        print(f"Test MSE: {test_mse}")

        r2 = 1 - np.sum(np.square(output_test - pred)) / np.sum(np.square(output_test - np.mean(output_test)))
        print(f"R^2: {r2}")

        return test_mse, r2

    def save(self, path: str, identifier: str = "_") -> None:
        """
        Save all models in the model_list to the specified path.

        Args:
            path (str): Directory to save the models.
            identifier (str): Identifier to append to the model names.
        """
        # Ensure the directory exists
        os.makedirs(path, exist_ok=True)

        # Save each model in the model_list with a unique identifier
        for index, model in enumerate(self.model_list):
            model_name = identifier + self.names[index]
            model.save(os.path.join(path, f'model_{model_name}.h5'))

    @classmethod
    def load(cls, path: str, identifier: str) -> 'MultiFidelity':
        """
        Load all models into the model_list from the specified path.

        Args:
            path (str): Directory from which to load the models.
            identifier (str): Identifier used in the model names.

        Returns:
            MultiFidelity: An instance of the MultiFidelity class with loaded models.
        """
        # Create an instance of MultiFidelity without training data
        instance = cls(names=[], params=[], data_train=[], output_train=[], N=[], n=[])

        # Load each model from the specified path
        for name in instance.names:
            model_path = os.path.join(path, f'model_{identifier + name}.h5')
            model = Neural_Network.load(model_path, identifier + name)
            instance.model_list.append(model)

        return instance


    def get_output(self) -> np.ndarray:
        """
        Get the final output from the outputs.

        Returns:
            np.ndarray: The last output.
        """
        return self.model_list[-1].prediction(self.data_train[-1])

In [ ]:
class LSTM_network(INetwork):

    def __init__(self, 
                 params: Optional[dict] = None, 
                 data_train: Optional[np.ndarray] = None, 
                 output_train: Optional[np.ndarray] = None, 
                 N: int = 1000, 
                 train: bool = True, 
                 do_HPO: bool = False, 
                 transformations: List[Any] = [], 
                 verbose: bool = False):
        """
        Initialize LSTM_network instance.

        Args:
            params (Optional[dict]): Hyperparameters for the network.
            data_train (Optional[np.ndarray]): Training data.
            output_train (Optional[np.ndarray]): Training outputs.
            N (int): Number of epochs for training.
            train (bool): Flag to indicate if training should be performed.
            do_HPO (bool): Flag to indicate if hyperparameter optimization is to be performed.
            transformations (List[Any]): List of transformations to apply to the data.
            verbose (bool): Flag to indicate verbosity of the output.
        """
        self.params = params
        self.name = "LSTM" 
        self.N = N
        self.verbose = verbose
        self.hist = None
        self.data_train = data_train
        self.output_train = output_train
        self.transformations = transformations

        self.input_shape = 1
        self.output_shape = 1
        self.inputs = None

        # Determine input and output shapes
        if data_train is not None and len(data_train.shape) > 1:
            self.input_shape = data_train.shape[-1]
        if output_train is not None and len(output_train.shape) > 1:
            self.output_shape = output_train.shape[-1]

        # Perform hyperparameter optimization if required
        if do_HPO or params is None:
            if output_train is None or data_train is None:
                warnings.warn("Not enough data given!", UserWarning)
            self.params = self.HPO(data_train, output_train)

        # Initialize the model
        self.model = getModel(self.params,self.input_shape,self.name,self.output_shape)  # dim_input = n_POD + 2, dim_output = n_POD

        # Train the model if required
        if train: 
            self.hist = self.training(int(params['sequence_length']),int(params['sequence_freq']),epoch=self.N) 
            self.plot_training_loss()
        else:
            name = './models/MF_POD_model'
            self.model = tf.keras.models.load_model(name) 


    @compute_time
    def training(self,seq_length,seq_freq,epoch):
        """
        Train the LSTM model.

        Args:
            seq_length (int): Sequence length for training.
            seq_freq (int): Sequence frequency for training.
            epoch (int): Number of epochs for training.

        Returns:
            tf.keras.callbacks.History: Training history.
        """
        self.sequence_length = seq_length
        self.sequence_freq = seq_freq
        self.input_train_seq, self.output_train_seq = _sliding_windows(data_train, output_train, self.sequence_length, self.sequence_freq)

        callback = tf.keras.callbacks.EarlyStopping(monitor='mse', patience=self.params['patience'], restore_best_weights=True)
        tf.keras.utils.set_random_seed(29)
        tf.config.experimental.enable_op_determinism() # for reproducibility

        self.hist = self.model.fit(self.input_train_seq, self.output_train_seq, epochs=epoch, verbose = self.verbose, callbacks=[callback])

        return self.hist

    def plot_training_loss(self) -> None:
        """
        Plots the training loss.
        """
        if self.hist is not None and 'loss' in self.hist.history:
            plt.plot(self.hist.history['loss'][100:], label='Training Loss')
            plt.title('Mean Squared Error (MSE) over Epochs')
            plt.xlabel('Epochs')
            plt.ylabel('MSE')
            plt.legend()
            plt.show()

    def prediction(self, x_test: np.ndarray) -> np.ndarray:
        """
        Make predictions using the LSTM model.

        Args:
            x_test (np.ndarray): Test data.

        Returns:
            np.ndarray: Predicted values.
        """
        if self.verbose:
            y_pred = self.model.predict(x_test)
        else:
            with Suppressor():
                y_pred = self.model.predict(x_test)
        return y_pred
        
    def performance(self, data_test: np.ndarray, output_test: np.ndarray, position: Optional[int] = None) -> Tuple[float, float]:
        """
        Evaluate the performance of the LSTM model.

        Args:
            data_test (np.ndarray): Test data.
            output_test (np.ndarray): Test outputs.
            position (Optional[int]): Position of the model in the model list.

        Returns:
            Tuple[float, float]: Mean Squared Error (MSE) and R-squared (R^2) values.
        """
        data = copy.copy(data_test)
        if position is None:
            position = len(self.model_list)
        elif not isinstance(position, int) or position > len(self.model_list):
            raise ValueError('The required NN does not exist.')

        # Iterate through the model list and make predictions
        for i in range(position - 1):
            data = np.concatenate((data, self.model_list[i].wrapper_prediction(data).reshape(-1, 1)), axis=1)

        # Predict using the specified model
        pred = self.model_list[position - 1].prediction(data)

        # Ensure the output shape matches the prediction shape
        if len(output_test.shape) < len(pred.shape):
            output_test = output_test[:, np.newaxis]
            
        test_mse = np.mean(np.square(output_test - pred))
        print(f"Test MSE: {test_mse}")

        r2 = 1 - np.sum(np.square(output_test - pred)) / np.sum(np.square(output_test - np.mean(output_test)))
        print(f"R^2: {r2}")

        return test_mse, r2

    def save(self, path: str, identifier: str) -> None:
        """
        Save the LSTM model and the class instance.

        Args:
            path (str): Directory path to save the model and instance.
            identifier (str): Identifier for the saved files.
        """
        # Ensure the directory exists
        os.makedirs(path, exist_ok=True)

        # Save the Keras model separately
        model_path = os.path.join(path, f'lstm_model_{identifier}.h5')
        self.model.save(model_path)

        # Save the class instance excluding the Keras model
        temp_model = self.model
        self.model = None
        with open(os.path.join(path, f'class_instance_{identifier}.pkl'), 'wb') as f:
            pickle.dump(self, f)

        # Restore the model attribute
        self.model = temp_model

    @classmethod
    def load(cls, path: str, identifier: str) -> 'LSTM_network':
        """
        Load the LSTM model and the class instance.

        Args:
            path (str): Directory path from which to load the model and instance.
            identifier (str): Identifier for the saved files.

        Returns:
            LSTM_network: The loaded LSTM_network instance.
        """
        with open(os.path.join(path, f'class_instance_{identifier}.pkl'), 'rb') as f:
            instance = pickle.load(f)

        model_path = os.path.join(path, f'lstm_model_{identifier}.h5')
        custom_objects = {
            'mse': MeanSquaredError()  # Add any custom objects required by the model
        }
        instance.model = load_model(model_path, custom_objects=custom_objects)

        return instance


    def _sliding_windows(self, data_input, data_output, seq_length, freq=1):
        """
        Generates sliding windows for the given data and labels.

        Args:
            data (np.ndarray): Input data.
            labels (np.ndarray): Output labels.
            seq_length (int): Length of each sequence.
            seq_freq (int): Frequency of each sequence.

        Returns:
            Tuple[np.ndarray, np.ndarray]: Input sequences and corresponding output sequences.
        """
    x = []
    y = []

    for i in range(data_input.shape[0]):
        for j in range(0, data_input.shape[1] - seq_length, freq):
            _x = data_input[i, j:(j + seq_length), :]
            _y = data_output[i, j:(j + seq_length), :]
            x.append(_x)
            y.append(_y)

    return np.array(x), np.array(y)



    def _input_wrapper_prediction(self, x_test: np.ndarray, multi_input: bool = False) -> np.ndarray:
        """
        Wrap the input for prediction by applying low-fidelity forward modeling.

        Args:
            x_test (np.ndarray): Test data.
            multi_input (bool): Flag indicating if multiple inputs are used.

        Returns:
            np.ndarray: Transformed prediction input.
        """
        # Call the parent class's _input_wrapper_prediction method
        x_final = super()._input_wrapper_prediction(x_test, multi_input)

        # Apply forward low-fidelity modeling to the wrapped input
        prediction_input = forward_low_fidelity(x_final)

        return prediction_input

In [ ]:
class Intermediate(INetwork):
    def __init__(self, 
                 params: Optional[dict] = None, 
                 data_train: Optional[List[np.ndarray]] = None, 
                 output_train: Optional[List[np.ndarray]] = None, 
                 N: int = 1000, 
                 n: int = 10, 
                 train: bool = True, 
                 do_HPO: bool = False, 
                 verbose: bool = False):
        """
        Initialize Intermediate network.

        Args:
            params (Optional[dict]): Parameters for the network.
            data_train (Optional[List[np.ndarray]]): Training data for the network.
            output_train (Optional[List[np.ndarray]]): Training outputs for the network.
            N (int): Number of epochs for training.
            n (int): Batch size for training.
            train (bool): Whether to train the model.
            do_HPO (bool): Whether to perform hyperparameter optimization.
            verbose (bool): Whether to print verbose output.
        """
        self.params = params
        self.name = "Inter" 
        self.N = N
        self.n = n
        self.verbose = verbose
        self.hist = None
        self.data_train = data_train
        self.output_train = output_train
        self.transformations = []
        
        self.input_shape = 1
        self.output_shape = 1

        if len(data_train) != 2 or len(output_train) != 2:
            raise ValueError('The data are incoherent or insufficient')
                
        # Concatenate data for training
        data_train = np.concatenate((data_train[1], data_train[0]), axis=0)
        output_train = np.concatenate((output_train[1], output_train[0]), axis=0)
        
        # Determine input and output shapes
        if data_train is not None and len(data_train.shape) > 1:
            self.input_shape = data_train.shape[1]

        if output_train is not None and len(output_train.shape) > 1:
            self.output_shape = output_train.shape[1]

        if do_HPO or params is None:
            if output_train is None or data_train is None:
                warnings.warn("Not enough data given!", UserWarning)
            self.params = self.HPO(data_train, output_train)

        # Create the model
        self.model = getModel(self.params, self.input_shape, self.name, self.output_shape)

        if train:
            self.hist = self.model.fit(data_train, output_train, epochs=self.N, batch_size=self.n, verbose=self.verbose) 
            self.plot_training_loss()

    @compute_time
    def training(self, x: np.ndarray, y: np.ndarray, epoch: int, batch: int) -> Any:
        """
        Train the model on the given data.

        Args:
            x (np.ndarray): Training data.
            y (np.ndarray): Training labels.
            epoch (int): Number of epochs.
            batch (int): Batch size.

        Returns:
            Any: Training history.
        """
        self.hist = self.model.fit(x, y, epochs=epoch, batch_size=batch, verbose=self.verbose) 
        return self.hist
    
    def plot_training_loss(self) -> None:
        """
        Plot the training loss.
        """
        if self.hist is not None and 'loss' in self.hist.history:
            plt.plot(self.hist.history['loss'][100:], label='Training Loss')
            plt.title('Mean Squared Error (MSE) over Epochs')
            plt.xlabel('Epochs')
            plt.ylabel('MSE')
            plt.legend()
            plt.show()

    def prediction(self, x_test: List[np.ndarray]) -> np.ndarray:
        """
        Make predictions using the model.

        Args:
            x_test (List[np.ndarray]): Test data.

        Returns:
            np.ndarray: Predictions.
        """
        if len(x_test) != 2:
            raise ValueError("Not enough data given")
        
        x_test = np.concatenate((x_test[1], x_test[0]), axis=0)

        if self.verbose:
            return self.model.predict(x_test)
        else:
            with Suppressor():
                return self.model.predict(x_test)

    def performance(self, data_test: List[np.ndarray], output_test: List[np.ndarray]) -> Tuple[float, float]:
        """
        Evaluate the performance of the model.

        Args:
            data_test (List[np.ndarray]): Test data.
            output_test (List[np.ndarray]): Test labels.

        Returns:
            Tuple[float, float]: Test MSE and R^2 score.
        """
        data_test = np.concatenate((data_test[1], data_test[0]), axis=0)
        output_test = np.concatenate((output_test[1], output_test[0]), axis=0)
        
        pred = self.prediction(data_test)

        if len(output_test.shape) < len(pred.shape):
            output_test = output_test[:, np.newaxis]

        test_mse = np.mean(np.square(output_test - pred))
        print(f"Test MSE: {test_mse}")

        r2 = 1 - np.sum(np.square(output_test - pred)) / np.sum(np.square(output_test - np.mean(output_test)))
        print(f"R^2: {r2}")
        
        return test_mse, r2

    def objective(self, par: dict) -> Dict[str, Any]:
        """
        Objective function for hyperparameter optimization.

        Args:
            par (dict): Hyperparameters.

        Returns:
            Dict[str, Any]: Result of cross-validation.
        """
        K.clear_session()
        CVres = kCrossVal(self.n, self.N, self.data_train, self.output_train, par, self.name, self.input_shape)
        return {"loss": CVres, "params": par, "status": STATUS_OK} 

    def HPO(self, data_train: np.ndarray, output_train: np.ndarray) -> dict:
        """
        Hyperparameter Optimization (to be implemented).

        Args:
            data_train (np.ndarray): Training data.
            output_train (np.ndarray): Training labels.

        Returns:
            dict: Best hyperparameters.
        """
        pass  # Implement hyperparameter optimization logic here

In [ ]:
@compute_time
    def inverse_cuqi(mean_prior: np.ndarray, 
                    x_data: np.ndarray,
                    x_real: Optional[np.ndarray] = None, 
                    y_obs: Optional[np.ndarray] = None, 
                    N: int = 1000, 
                    burn_in: int = 500, 
                    cov_prior: float = 0.5, 
                    sd_noise: float = 0.1,
                    adapt: bool = False, 
                    scale: float = 0.3, 
                    proposal_sd: float = 0.3, 
                    x_init: Optional[Union[int, float, np.ndarray]] = None, 
                    diagnostic: bool = True, 
                    number_chains: int = 1, 
                    algo: str = "MH", 
                    transformation: List = []) -> Union[np.ndarray, float]:
        """
        Perform Bayesian inference using the CUQI framework.

        Parameters:
        - mean_prior: np.ndarray : Prior mean
        - x_data: np.ndarray : Input data
        - x_real: Optional[np.ndarray] : Real data (optional)
        - y_obs: Optional[np.ndarray] : Observed data (optional)
        - N: int : Number of iterations (default: 1000)
        - burn_in: int : Number of burn-in steps (default: 500)
        - cov_prior: float : Covariance of the prior (default: 0.5)
        - sd_noise: float : Standard deviation of noise (default: 0.1)
        - adapt: bool : Whether to adapt the proposal distribution (default: False)
        - scale: float : Scaling factor for the proposal distribution (default: 0.3)
        - proposal_sd: float : Standard deviation of the proposal distribution (default: 0.3)
        - x_init: Optional[Union[int, float, np.ndarray]] : Initial guess (default: None)
        - diagnostic: bool : Whether to plot diagnostic information (default: True)
        - number_chains: int : Number of MCMC chains (default: 1)
        - algo: str : Algorithm to use ("MH" or "NUTS", default: "MH")
        - transformation: List : List of transformations (default: [])

        Returns:
        - estimates: np.ndarray : Estimated parameters
        - error: float : Error with respect to true parameters
        """

        # Set inputs and transformations
        self.inputs = x_data
        self.transformations = transformation

        # Check if observations are provided
        if y_obs is not None:
            dim = y_obs.shape[0]
        else:
            warning_message = "No observation nor data given"
            warnings.warn(warning_message, UserWarning)
            return

        # Check if the number of steps is greater than burn-in period
        if N <= burn_in:
            warning_message = "Number of steps insufficient, smaller or equal than burn-in"
            warnings.warn(warning_message, UserWarning)
            return

        # Initialize x_init if not provided
        if x_init is None:
            x_init = np.random.rand(dim)
        elif isinstance(x_init, (int, float)):
            x_init = x_init * np.ones(dim)
        
        m = x_real.shape[0]
        
        # Select algorithm and initialize CuqiModel and Gaussian objects
        if algo == "NUTS":
            fun = Function(wrapper_prediction)
            A = CuqiModel(forward=wrapper_prediction, jacobian=fun.compute_jacobian, range_geometry=Continuous1D(dim), domain_geometry=Continuous1D(dim))
            x = Gaussian(mean=mean_prior, cov=cov_prior)
        else:
            A = CuqiModel(forward=wrapper_prediction, range_geometry=Continuous1D(dim), domain_geometry=Continuous1D(m))
            x = Gaussian(mean=mean_prior, cov=cov_prior)

        y = Gaussian(mean=A(x), cov=proposal_sd)

        # Generate or perturb observations
        if y_obs is None:
            y_obs = y(x=x_real).sample()
        else:
            y_obs = y_obs + np.random.normal(loc=0., scale=sd_noise, size=y_obs.shape)

        # Run MCMC to get estimates
        estimates = MCMC_cuqi(y, x, y_obs, N, burn_in, number_chains, diagnostic=diagnostic, algo=algo, adapt=adapt, scale=scale)
        estimates = np.mean(estimates, axis=1)

        # Calculate and print error
        error = np.linalg.norm(estimates - x_real)
        print(f"Error wrt true parameters: {error}")

        # Plot diagnostics if required
        if diagnostic:
            plot_hist(estimates, x_real, wrapper_prediction(estimates), wrapper_prediction(x_real))

        return estimates, error



In [ ]:

def MCMC_cuqi(
    y: Any, 
    x: Any, 
    observation: np.ndarray, 
    N: int, 
    burn_in: int, 
    n: int = 1, 
    diagnostic: bool = True, 
    algo: str = "MH", 
    adapt: bool = False, 
    scale: float = 0.3
) -> np.ndarray:
    """
    Perform MCMC sampling using CUQI library.

    Args:
        y (Any): Observed data.
        x (Any): Model inputs.
        observation (np.ndarray): Observed values.
        N (int): Number of samples to draw.
        burn_in (int): Number of initial samples to discard.
        n (int, optional): Number of chains. Default is 1.
        diagnostic (bool, optional): Whether to produce diagnostic plots. Default is True.
        algo (str, optional): Sampling algorithm to use. Options are 'MH', 'NUTS', 'pCN'. Default is 'MH'.
        adapt (bool, optional): Whether to adapt the sampler. Default is False.
        scale (float, optional): Scaling factor for the Metropolis-Hastings algorithm. Default is 0.3.

    Returns:
        np.ndarray: Estimates of the parameters.
    """
    
    dim = observation.shape[0]
    x_init = np.random.rand(dim)
    estimates = np.empty((1, 0))    # Initialize parameter estimates
    chains = np.empty((0, n, N - burn_in)) 
    post = np.empty((1, 0))         # Placeholder for posterior samples

    posterior = JointDistribution(y, x)(y=observation)

    # Select the MCMC sampler based on the algorithm
    samplers = {
        "NUTS": lambda: NUTS(posterior, x0=x_init),
        "MH": lambda: MH(posterior, scale=scale) if not adapt else MH(posterior),
        "pCN": lambda: pCN(posterior, x0=x_init)
    }

    if algo not in samplers:
        raise ValueError(f"Unknown algorithm {algo}")

    sampler = samplers[algo]()
    
    # Sample from the posterior distribution
    samples = sampler.sample_adapt(N - burn_in, burn_in) if adapt else sampler.sample(N - burn_in, burn_in)

    estimates = np.column_stack((estimates, samples.mean()[:, np.newaxis]))
    chains = np.concatenate((chains, np.expand_dims(samples.samples, axis=0)), axis=0)
    post = np.concatenate((post, samples.samples), axis=1)

    print(f"Mean values = {estimates.mean(axis=1)}")

    # Plot trace plots
    for l in range(chains.shape[1]):
        plt.figure(figsize=(10, 4))
        for i in range(chains.shape[0]):
            plt.plot(chains[i, l, :], label=f'Chain {i+1}')
        plt.xlabel('Sample')
        plt.ylabel('Value')
        plt.title(f'Trace Plot for variable {l}')
        plt.legend()
        plt.show()

    # Diagnostic plots
    if diagnostic:
        if n == 1:
            samples.plot_trace()
            samples.plot_autocorrelation()
        else:
            num_bins = 20
            plt.figure()
            for num in range(post.shape[0]):
                bin_edges = np.linspace(np.min(post[num, :]), np.max(post[num, :]), num_bins + 1)
                hist, _ = np.histogram(post[num, :], bins=bin_edges)
                hist = hist / post.shape[1]
                bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
                plt.bar(bin_centers, hist, width=np.diff(bin_edges), edgecolor='black', label=f'Var {num + 1}')
                plt.xlabel('Value')
                plt.ylabel('Probability')
                plt.title('Posterior Distribution')
                plt.legend()
                plt.show()

            autocov = az.autocov(chains[:, 0, :])
            ess = az.ess(chains[:, 0, :])
            print(f"Effective Sample Size (ESS) values = {ess}")
            plt.figure()
            plt.plot(autocov[0, :])
            plt.title('Autocovariance of the first chain')
            plt.xlabel('Lag')
            plt.ylabel('Autocovariance')
            plt.legend()
            plt.show()

    return estimates

In [ ]:
def MCMC_cuqi(
    y: Any, 
    x: Any, 
    observation: np.ndarray, 
    N: int, 
    burn_in: int, 
    n: int = 1, 
    diagnostic: bool = True, 
    algo: str = "MH", 
    adapt: bool = False, 
    scale: float = 0.3
) -> np.ndarray:
    """
    Perform MCMC sampling using CUQI library.

    Args:
        y (Any): Dependent variable.
        x (Any): Independent variable.
        observation (np.ndarray): Observed data.
        N (int): Number of samples to draw.
        burn_in (int): Number of burn-in samples to discard.
        n (int, optional): Number of chains. Defaults to 1.
        diagnostic (bool, optional): Whether to plot diagnostic plots. Defaults to True.
        algo (str, optional): Sampling algorithm to use. Defaults to "MH".
        adapt (bool, optional): Whether to use adaptive sampling. Defaults to False.
        scale (float, optional): Scaling factor for MH algorithm. Defaults to 0.3.

    Returns:
        np.ndarray: Array of estimated parameter means.
    """
    dim = observation.shape[0]
    x_init = np.random.rand(dim)
    estimates = np.empty((dim, 0))  # Dimensions of parameters to estimate
    chains = np.empty((0, dim, N - burn_in)) 
    post = np.empty((dim, 0))         
    posterior = JointDistribution(y, x)(y=observation)

    # Loop through the number of chains
    for i in range(n):
        # Select the appropriate sampler
        if algo == "NUTS":
            sampler = NUTS(posterior, x0=x_init)
        elif algo == "MH":
            sampler = MH(posterior, scale=scale) if not adapt else MH(posterior)
        elif algo == "pCN":
            sampler = pCN(posterior, x0=x_init)
        else:
            raise ValueError(f"Unknown algorithm {algo}")

        # Perform sampling
        samples = sampler.sample_adapt(N - burn_in, burn_in) if adapt else sampler.sample(N - burn_in, burn_in)

        # Compute estimates and store chains
        estimates = np.column_stack((estimates, samples.mean()[:, np.newaxis]))
        chains = np.concatenate((chains, np.expand_dims(samples.samples, axis=0)), axis=0)
        post = np.concatenate((post, samples.samples), axis=1)

        print(f"Chain {i+1} mean values: {estimates.mean(axis=1)}")

    # Plot trace plots for each parameter
    for l in range(chains.shape[1]):
        plt.figure(figsize=(10, 4))
        for i in range(chains.shape[0]):
            plt.plot(chains[i, l, :])
        plt.xlabel('Sample')
        plt.ylabel('Value')
        plt.title(f'Trace Plot for variable {l}')
        plt.legend([f'Chain {i+1}' for i in range(chains.shape[0])])
        plt.show()

    # Diagnostic plots
    if diagnostic:
        if n == 1:
            cuqi_plot.trace(samples)
            cuqi_plot.autocorrelation(samples)
        else:
            # Plot histograms for the posterior distributions
            num_bins = 20
            for num in range(post.shape[0]):
                plt.figure()
                bin_edges = np.linspace(np.min(post[num, :]), np.max(post[num, :]), num_bins + 1)
                hist, _ = np.histogram(post[num, :], bins=bin_edges)
                hist = hist / post.shape[1]
                bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
                plt.bar(bin_centers, hist, width=np.diff(bin_edges), edgecolor='black', label=f'Var {num + 1}')
                plt.xlabel('Value')
                plt.ylabel('Probability')
                plt.title(f'Distribution of variable {num}')
                plt.legend()
                plt.show()

            # Plot autocovariance and ESS for the first chain
            autocov = az.autocov(chains[:, 0, :])
            ess = az.ess(chains[:, 0, :])
            print(f"Effective Sample Size (ESS): {ess}")
            plt.figure()
            plt.plot(autocov[0, :])
            plt.title('Autocovariance of the first chain')
            plt.xlabel('Lag')
            plt.ylabel('Autocovariance')
            plt.legend(['Autocovariance'])
            plt.show()

    return estimates

In [ ]:
    for i in range(n):
        # Select the appropriate sampler
        if algo == "NUTS":
            sampler = NUTS(posterior, x0=x_init)
        elif algo == "MH":
            sampler = MH(posterior, scale=scale) if not adapt else MH(posterior)
        elif algo == "pCN":
            sampler = pCN(posterior, x0=x_init)
        else:
            raise ValueError(f"Unknown algorithm {algo}")

        # Perform sampling
        samples = sampler.sample_adapt(N - burn_in, burn_in) if adapt else sampler.sample(N - burn_in, burn_in)

        # Compute estimates and store chains
        estimates = np.column_stack((estimates, samples.mean()[:, np.newaxis]))
        chains = np.concatenate((chains, np.expand_dims(samples.samples, axis=0)), axis=0)
        post = np.concatenate((post, samples.samples), axis=1)

        print(f"Chain {i+1} mean values: {estimates.mean(axis=1)}")


In [ ]:
import ray

ray.init()

context = get_runtime_context()
worker_id = ray.context.worker_id.hex()  # Get worker ID as a hex string

print(f"Worker {worker_id} is running chain {chain_id}")

In [ ]:
import ray

# Initialize Ray
ray.init()

# Define a Ray remote function
@ray.remote
def run_markov_chain_parallel(chain_id, num_steps):
    return run_markov_chain(chain_id, num_steps)

def run_chains(parallel, num_chains, num_steps):
    if parallel:
        # Parallel execution using Ray
        futures = [run_markov_chain_parallel.remote(chain_id, num_steps) for chain_id in range(num_chains)]
        results = ray.get(futures)
    else:
        # Sequential execution
        results = [run_markov_chain(chain_id, num_steps) for chain_id in range(num_chains)]
    
    return results

# Example usage
parallel = True  # Change this to False for sequential execution
num_chains = 10
num_steps = 1000

results = run_chains(parallel, num_chains, num_steps)

for chain_id, states in results:
    print(f"Chain {chain_id}: Final State = {states[-1]}")

# Shutdown Ray
ray.shutdown()
